# 5.7 — Train MuscleMimic from scratch with FastTD3

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MyoHub/myosuite/blob/tutorial/fasttd3-directional-scratch/tutorials/5.7_FastTD3_Directional_From_Scratch.ipynb)

Tutorial 5.3 trains a MuscleMimic walker with **on-policy PPO** on mjlab. This notebook trains the same full-body body **from scratch** with **off-policy FastTD3** (Seo et al., 2025) — no motion clip, no Hugging Face token, no behaviour cloning.

The task is `myoMimicFullbody-v0`: 354 muscles, random site targets. FastTD3's actor outputs `tanh` actions in `[-1, 1]`; the trainer maps those to muscle activations in `[0, 1]`.

## Flow

1. Install MyoSuite and the FastTD3 trainer (`training/fasttd3`, not the `myosuite` package).
2. Open `myoMimicFullbody-v0` and print observation / action sizes.
3. Run a few CPU steps to check the loop (no GPU, no Warp).
4. Optionally train on a Colab T4 with the mjlab Warp twin.

The CPU cell is the default. Set `GPU_TRAIN = True` only after Colab shows a GPU (Runtime → Change runtime type → T4).

A 50k-step T4 run is a demo, not a finished walker. For a serious run use a larger GPU and more environments (see `training/fasttd3/README.md`).

> **Citation and data:** FastTD3 is [younggyoseo/FastTD3](https://github.com/younggyoseo/FastTD3) (Seo et al., 2025, MIT). MuscleMimic: [Li et al., 2026](https://arxiv.org/abs/2603.25544). Motions in later tutorials come from [AMASS](https://amass.is.tue.mpg.de) / KIT, retargeted with GMR; this notebook does **not** download those clips. References: the MyoSuite documentation *References* page.

## 5.7.0 — Config

`GPU_TRAIN` stays `False` until the runtime actually has CUDA. Leave it off for the CPU smoke; that cell does not need Warp or a GPU.

In [ ]:
ENV_ID = "myoMimicFullbody-v0"
GPU_TRAIN = False  # True only on a T4/A100 runtime
TRAIN_STEPS = 50_000
N_ENVS = 64  # T4; 1024 typically OOMs this model
print(f"env={ENV_ID}  GPU_TRAIN={GPU_TRAIN}  steps={TRAIN_STEPS}  envs={N_ENVS}")

## 5.7.1 — Setup

On Colab this clones MyoSuite, installs `myosuite[musclemimic]` and `training/fasttd3`, and keeps the runtime's existing PyTorch build (so a GPU wheel is not replaced by a CPU one).

Locally it only needs a MyoSuite checkout with those extras already installed:

```bash
pip install -e ".[musclemimic]"
pip install -e training/fasttd3
```

In [ ]:
from __future__ import annotations

import os
import subprocess
import sys
from pathlib import Path

GIT_URL = "https://github.com/MyoHub/myosuite.git"
GIT_REF = "tutorial/fasttd3-directional-scratch"

os.environ["HF_HUB_OFFLINE"] = "1"
os.environ.setdefault("JAX_PLATFORMS", "cpu")

try:
    import google.colab  # noqa: F401

    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if sys.platform != "darwin":
    os.environ["MUJOCO_GL"] = "egl"
    os.environ.setdefault("PYOPENGL_PLATFORM", "egl")

_nb = globals().get("__vsc_ipynb_file__")
if _nb:
    os.chdir(os.path.dirname(_nb))


def _pip(*pkgs: str) -> None:
    cmd = [
        sys.executable, "-m", "pip", "install", "-q",
        "--upgrade-strategy", "only-if-needed",
        "--constraint", str(_KEEP_TORCH), *pkgs,
    ]
    subprocess.check_call(cmd)


_KEEP_TORCH = Path("/tmp/keep-torch.txt")
try:
    import torch as _pre_torch

    _KEEP_TORCH.write_text(f"torch=={_pre_torch.__version__.split('+')[0]}\n")
except ImportError:
    _KEEP_TORCH.write_text("")

if IN_COLAB:
    subprocess.run(["apt-get", "-qq", "update"], check=False)
    subprocess.run(
        ["apt-get", "-qq", "install", "-y", "libegl1", "libgles2", "libosmesa6"],
        check=False,
    )
    dest = Path("/content/myosuite")
    if dest.exists():
        subprocess.check_call(["git", "-C", str(dest), "fetch", "--depth", "1", "origin", GIT_REF])
        subprocess.check_call(["git", "-C", str(dest), "checkout", "-B", GIT_REF, "FETCH_HEAD"])
    else:
        subprocess.check_call(
            ["git", "clone", "--depth", "1", "--branch", GIT_REF, GIT_URL, str(dest)]
        )
    os.chdir(dest)
    if str(dest) not in sys.path:
        sys.path.insert(0, str(dest))
    _pip("-e", f"{dest}[musclemimic]")
    _pip("-e", str(dest / "training" / "fasttd3"))
    REPO = dest
else:
    here = Path.cwd().resolve()
    REPO = next(
        (d for d in (here, *here.parents) if (d / "myosuite" / "__init__.py").is_file()),
        None,
    )
    if REPO is None:
        raise FileNotFoundError("Run this notebook from a myosuite clone, or open it in Colab.")
    if str(REPO) not in sys.path:
        sys.path.insert(0, str(REPO))
    pkg = REPO / "training" / "fasttd3"
    if str(pkg) not in sys.path:
        sys.path.insert(0, str(pkg))

import torch

OUT = REPO / "runs" / "fasttd3"
print("repo:", REPO)
print(f"torch {torch.__version__}  cuda={torch.cuda.is_available()}  colab={IN_COLAB}")

## 5.7.2 — The environment

`myoMimicFullbody-v0` is the MuscleMimic full-body model (`musclemimic_models` 1.0.6) with **random site targets**. That is the same skeleton ChaseTag and tutorial 5.4 use, but this cell does not load KIT/AMASS clips, so it does not need a Hugging Face token.

MyoSuite's own `myo_sim` full-body composition is a fallback with the same DoF count and muscle curves, but the clavicles sit centimetres away — checkpoints and ChaseTag policies will not transfer. Stay on `musclemimic_models` here.

Observation is on the order of 684 values (qpos, qvel, muscle activations, site positions, target). Action is 354 muscle excitations.

In [ ]:
from myosuite.core.registry import make_env

env = make_env(ENV_ID)
obs, info = env.reset(seed=0)
print("observation", getattr(obs, "shape", None) or len(obs))
print("action     ", env.action_space.shape)
print("action low/high", float(env.action_space.low.min()), float(env.action_space.high.max()))
env.close()

## 5.7.3 — FastTD3 in this repository

The trainer is a **separate project** under `training/fasttd3/` ([placement guidelines](https://github.com/MyoHub/myosuite/pull/581)). It is not an extra on the installable `myosuite` package.

| Piece | Where |
| --- | --- |
| Official Actor / distributional Critic / replay | [younggyoseo/FastTD3](https://github.com/younggyoseo/FastTD3) |
| MyoSuite CPU and mjlab wrappers | `training/fasttd3/myosuite_fasttd3/env.py` |
| Update loop | `python -m myosuite_fasttd3` |

The actor is the published FastTD3 `tanh` policy. Muscle tasks only remap the action in the env wrapper. Support for the categorical critic is `[-250, 250]` (HumanoidBench defaults). AMP stays off on T4: that GPU has no bf16, and fp16 overflows on unnormalized muscle observations.

## 5.7.4 — CPU smoke

Two Gymnasium environments, eight steps, tiny networks. This only checks that the official FastTD3 modules import, the env steps, and a checkpoint is written. It will not learn to walk.

Expect a line like `FastTD3 env=myoMimicFullbody-v0 backend=cpu ... amp=False` and a final `saved ... smoke.pt`.

In [ ]:
cmd = [
    sys.executable, "-m", "myosuite_fasttd3", ENV_ID,
    "--smoke", "--backend", "cpu", "--device", "cpu",
    "--out", str(OUT),
]
print(" ".join(cmd), flush=True)
subprocess.check_call(cmd, cwd=str(REPO))
print("CPU smoke ok")

## 5.7.5 — GPU training (optional)

Uses the mjlab Warp twin of the same env id. On Colab:

1. Runtime → Change runtime type → Hardware accelerator **T4 GPU**.
2. Runtime → Disconnect and delete runtime, reopen, run from 5.7.0.
3. Set `GPU_TRAIN = True` in 5.7.0 and run this cell.

If this session has no CUDA, the cell prints `[SKIP]` instead of exiting. 64 environments fit a T4; 1024 usually does not.

Log lines look like `t=... fps=... qloss=... q=... pi=... r=...`. `q` is the distributional value and should stay inside about `[-250, 250]`. `r` is the mean step reward (roughly 0–1 on this task). A run that prints `amp=True` or `buf=` / `upd=` is not this trainer.

In [ ]:
if not GPU_TRAIN:
    print("[SKIP] GPU_TRAIN is False. Run the CPU smoke, then enable this on a T4 runtime.")
elif not torch.cuda.is_available():
    print(
        "[SKIP] no CUDA in this session "
        f"(torch={torch.__version__}). "
        "Colab: Runtime → Change runtime type → T4 GPU, disconnect, rerun from 5.7.0."
    )
else:
    if IN_COLAB:
        _pip("mjlab>=1.4.0", "mujoco-warp>=3.7.0", "warp-lang>=1.12.0")
    cmd = [
        sys.executable, "-m", "myosuite_fasttd3", ENV_ID,
        "--backend", "mjlab",
        "--num-envs", str(N_ENVS),
        "--total-timesteps", str(TRAIN_STEPS),
        "--out", str(OUT),
    ]
    print(" ".join(cmd), flush=True)
    subprocess.check_call(cmd, cwd=str(REPO))
    print("GPU train ok")

## 5.7.6 — Next

- Command line, from a clone: `python -m myosuite_fasttd3 myoMimicFullbody-v0 --backend mjlab --num-envs 1024`
- Heading-conditioned walking with a teacher: tutorial **5.4** (`myoFullBodyDirectional-v0`; needs gated KIT clips).
- Submit a walker to ChaseTag: tutorial **5.6**.
- Trainer docs and attribution: `training/fasttd3/README.md`.